# Lottery tickets × warmup: Colab runner

Runs the experiments of `src/CLAUDE_1.md` from a copy of the project on Google Drive.
Read `NOTES.md` first: it has the current status, open decisions and the run order.

- **GPU runtime:** Runtime → Change runtime type → GPU (G4 / A100 / L4 / T4). bf16 or fp16 is picked automatically.
- **Resumable:** every finished training writes its JSON and checkpoint to Drive. If the session dies,
  re-run the setup cells and the same command; finished trainings are skipped.
- **Never run two training cells at once.**

In [ ]:
# 1. Mount Drive and go to the project folder (edit PROJECT if you put it elsewhere)
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/final_project'
%cd {PROJECT}

In [ ]:
# 2. Environment: Colab's preinstalled torch is used (do NOT pip install requirements.txt, it pins Windows cu128 wheels)
import os, torch, sys
os.environ['LTH_RAW_DATA'] = '/content/cifar_raw'   # raw CIFAR-10 on fast local disk, re-downloaded each session (~1 min)
os.environ['LTH_BUDGET_H'] = '18'                   # GPU-hour budget for this GPU type: EDIT to what you agreed (NOTES.md, O4)
print(sys.version, torch.__version__, torch.version.cuda)
print(torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0))
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv
!pip -q install pyyaml pytest   # usually already present

In [ ]:
# 3. Tests (~1 min)
!python -m pytest -q

In [ ]:
# 4. Benchmark: smoke run (section 7 step 1). Output also goes to results/smoke/seed0/log.txt
!python -m src.imp configs/smoke.yaml

In [ ]:
# 5. Benchmark summary: minutes per 15k training incl. sharpness, peak VRAM -> put in NOTES.md
import json, glob
for f in sorted(glob.glob('results/smoke/seed0/round*.json')):
    r = json.load(open(f)); n = len(r['sharpness'])
    step_ms = (r['wall_s'] - r['sharpness_s']) / r['iters'] * 1000
    meas_s = r['sharpness_s'] / n
    est_min = (15000 * step_ms / 1000 + 21 * meas_s) / 60
    print(f"{f.split('/')[-1]:24s} {step_ms:5.1f} ms/step  {meas_s:5.1f} s/lambda  "
          f"-> ~{est_min:4.1f} min per 15k run (21 lambda points), peak {r['peak_vram_mb']:.0f} MB, amp {r['amp_dtype']}")

## Phase A (only after Gate 0 in NOTES.md: sharpness-cost decision taken and smoke benchmark recorded)
One cell per chain; each is resumable. Order from section 7.

In [ ]:
!python -m src.imp configs/conv4_fmnist_high.yaml

In [ ]:
!python -m src.imp configs/resnet20_low.yaml

In [ ]:
!python -m src.imp configs/resnet20_high.yaml

In [ ]:
!python -m src.imp configs/resnet20_warm03.yaml

In [ ]:
!python -m src.imp configs/resnet20_high_warm.yaml

In [ ]:
# Gate A check + all figures/tables from results/
!python -m analysis.plots
!cat results/gate_a.md

## Phase B (only if Gate A passed)

In [ ]:
!python -m src.imp configs/resnet20_low.yaml --seeds 1
!python -m src.imp configs/resnet20_high.yaml --seeds 1
!python -m src.imp configs/resnet20_warm03.yaml --seeds 1

In [ ]:
# H4: masks from the warm03 chain held fixed
!python -m src.fixed_mask configs/h4_high.yaml
!python -m src.fixed_mask configs/h4_high_sam.yaml
!python -m src.fixed_mask configs/h4_high_anchor.yaml --select-lam
!python -m src.fixed_mask configs/h4_high_anchor.yaml
!python -m analysis.plots